In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple
import json
import math
import os
import random
import re

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm


@dataclass
class Config:
    dataset_name: str = "cbis_ddsm"
    output_dir: Path = Path("outputs/ceinet_s")
    seed: int = 2026
    image_size: int = 224
    batch_size: int = 8
    num_workers: int = 0
    epochs: int = 100
    learning_rate: float = 1e-4
    weight_decay: float = 1e-5
    early_stopping_patience: int = 20
    early_stopping_delta: float = 1e-4
    validation_fraction: float = 0.15
    test_fraction: float = 0.20
    split_level: str = "patient"
    pretrained_backbone: bool = True
    backbone_variant: str = "small"
    num_classes: int = 2
    num_tokens: int = 8
    token_candidate_factor: int = 6
    foreground_threshold: float = 0.02
    minimum_foreground_coverage: float = 0.02
    feedback_steps: int = 2
    evidence_floor: float = 1e-4
    conflict_gamma_init: float = 2.0
    kl_weight: float = 1.0
    kl_anneal_epochs: int = 10
    auxiliary_view_weight: float = 0.25
    gate_calibration_weight: float = 0.05
    use_augmentation: bool = True
    amp: bool = True
    gradient_clip_norm: float = 1.0
    ece_bins: int = 15
    evaluate_missing_views: bool = True
    vindr_image_root: Path = Path("data/VinDr-Mammo")
    vindr_annotations_csv: Path = Path("data/vindr_detection_v1_folds.csv")
    vindr_source_split: Optional[str] = None
    vindr_positive_birads_threshold: int = 4
    cbis_root: Path = Path("data/CBIS-DDSM")
    cbis_train_csv: Optional[Path] = None
    cbis_test_csv: Optional[Path] = None
    cbis_dicom_info_csv: Optional[Path] = None
    cbis_image_source: str = "dicom"


cfg = Config()


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def config_to_dict(config: Config) -> Dict[str, object]:
    values = asdict(config)
    return {key: str(value) if isinstance(value, Path) else value for key, value in values.items()}


def canonical_dataset_name(name: str) -> str:
    value = str(name).lower().strip().replace("-", "_")
    aliases = {
        "vindr": "vindr",
        "vindr_mammo": "vindr",
        "cbis": "cbis_ddsm",
        "cbis_ddsm": "cbis_ddsm",
    }
    if value not in aliases:
        raise ValueError("dataset_name must be 'vindr' or 'cbis_ddsm'")
    return aliases[value]


def resolve_cbis_paths(config: Config) -> Tuple[Path, Path, Path]:
    csv_dir = Path(config.cbis_root) / "csv"
    train_csv = Path(config.cbis_train_csv) if config.cbis_train_csv is not None else csv_dir / "mass_case_description_train_set.csv"
    test_csv = Path(config.cbis_test_csv) if config.cbis_test_csv is not None else csv_dir / "mass_case_description_test_set.csv"
    dicom_info_csv = Path(config.cbis_dicom_info_csv) if config.cbis_dicom_info_csv is not None else csv_dir / "dicom_info.csv"
    return train_csv, test_csv, dicom_info_csv


def validate_config(config: Config) -> Config:
    config.dataset_name = canonical_dataset_name(config.dataset_name)
    config.output_dir = Path(config.output_dir)
    config.vindr_image_root = Path(config.vindr_image_root)
    config.vindr_annotations_csv = Path(config.vindr_annotations_csv)
    config.cbis_root = Path(config.cbis_root)
    config.cbis_train_csv = Path(config.cbis_train_csv) if config.cbis_train_csv is not None else None
    config.cbis_test_csv = Path(config.cbis_test_csv) if config.cbis_test_csv is not None else None
    config.cbis_dicom_info_csv = Path(config.cbis_dicom_info_csv) if config.cbis_dicom_info_csv is not None else None
    config.split_level = str(config.split_level).lower().strip()

    if config.split_level not in {"patient", "study"}:
        raise ValueError("split_level must be 'patient' or 'study'")
    if config.dataset_name == "cbis_ddsm" and config.split_level != "patient":
        raise ValueError("CBIS-DDSM must use patient-level splitting")
    if config.backbone_variant not in {"tiny", "small"}:
        raise ValueError("backbone_variant must be 'tiny' or 'small'")
    if config.num_tokens < 1:
        raise ValueError("num_tokens must be at least one")
    if not 0.0 < config.test_fraction < 1.0:
        raise ValueError("test_fraction must be between zero and one")
    if not 0.0 < config.validation_fraction < 1.0:
        raise ValueError("validation_fraction must be between zero and one")

    if config.dataset_name == "vindr":
        if not config.vindr_image_root.exists():
            raise FileNotFoundError(f"VinDr image root was not found: {config.vindr_image_root}")
        if not config.vindr_annotations_csv.exists():
            raise FileNotFoundError(f"VinDr annotation CSV was not found: {config.vindr_annotations_csv}")
    else:
        if not config.cbis_root.exists():
            raise FileNotFoundError(f"CBIS-DDSM root was not found: {config.cbis_root}")
        for path in resolve_cbis_paths(config):
            if not path.exists():
                raise FileNotFoundError(f"CBIS-DDSM file was not found: {path}")

    return config

In [ ]:
def normalize_image(image: np.ndarray) -> np.ndarray:
    image = np.asarray(image, dtype=np.float32)
    image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
    low = float(image.min())
    high = float(image.max())
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)
    return ((image - low) / (high - low)).astype(np.float32)


def read_image(path: Path) -> np.ndarray:
    suffix = path.suffix.lower()
    if suffix in {".dcm", ".dicom"} or not suffix:
        dataset = pydicom.dcmread(str(path), force=True)
        image = dataset.pixel_array.astype(np.float32)
        slope = float(getattr(dataset, "RescaleSlope", 1.0))
        intercept = float(getattr(dataset, "RescaleIntercept", 0.0))
        image = normalize_image(image * slope + intercept)
        if str(getattr(dataset, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            image = 1.0 - image
        return image.astype(np.float32)
    return normalize_image(np.asarray(Image.open(path).convert("L"), dtype=np.float32))


def largest_component(mask: np.ndarray) -> np.ndarray:
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if count <= 1:
        return mask.astype(np.uint8)
    index = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return (labels == index).astype(np.uint8)


def crop_breast(image: np.ndarray, size: int) -> np.ndarray:
    image = normalize_image(image)
    image_u8 = np.clip(image * 255.0, 0.0, 255.0).astype(np.uint8)
    blurred = cv2.GaussianBlur(image_u8, (5, 5), 0)
    _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    border = np.concatenate([mask[0], mask[-1], mask[:, 0], mask[:, -1]])
    if float(border.mean()) > 160.0:
        mask = cv2.bitwise_not(mask)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), dtype=np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), dtype=np.uint8))
    mask = largest_component(mask > 0)

    coordinates = np.argwhere(mask > 0)
    if coordinates.size == 0:
        crop = image
    else:
        y0, x0 = coordinates.min(axis=0)
        y1, x1 = coordinates.max(axis=0) + 1
        height = y1 - y0
        width = x1 - x0
        padding = int(round(0.08 * max(height, width)))
        y0 = max(0, y0 - padding)
        x0 = max(0, x0 - padding)
        y1 = min(image.shape[0], y1 + padding)
        x1 = min(image.shape[1], x1 + padding)
        crop = image[y0:y1, x0:x1]

    height, width = crop.shape
    side = max(height, width)
    square = np.zeros((side, side), dtype=np.float32)
    top = (side - height) // 2
    left = (side - width) // 2
    square[top:top + height, left:left + width] = crop
    return cv2.resize(square, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32)


def foreground_area(image: np.ndarray) -> float:
    return float((image > 0.02).mean())


def normalize_column_name(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def find_column(frame: pd.DataFrame, candidates: Sequence[str], required: bool = True) -> Optional[str]:
    lookup = {normalize_column_name(column): column for column in frame.columns}
    for candidate in candidates:
        column = lookup.get(normalize_column_name(candidate))
        if column is not None:
            return column
    if required:
        raise KeyError(f"Missing one of: {list(candidates)}")
    return None


def find_image_path_column(frame: pd.DataFrame) -> str:
    for column in frame.columns:
        name = normalize_column_name(column)
        if "image" in name and "path" in name:
            return column
    raise KeyError("No image-path column was found")


def normalize_view(value: object) -> Optional[str]:
    value = str(value).upper().strip()
    if "MLO" in value:
        return "MLO"
    if "CC" in value:
        return "CC"
    return None


def parse_birads(value: object) -> Optional[int]:
    if pd.isna(value):
        return None
    match = re.search(r"\d+", str(value))
    return int(match.group()) if match else None


def pathology_label(value: object) -> int:
    text = str(value).upper().strip()
    if "MALIGNANT" in text:
        return 1
    if "BENIGN" in text:
        return 0
    raise ValueError(f"Unsupported pathology label: {value}")


def build_image_index(image_root: Path) -> Dict[str, Path]:
    extensions = {".png", ".jpg", ".jpeg", ".dcm", ".dicom"}
    index: Dict[str, Path] = {}

    for path in image_root.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix and path.suffix.lower() not in extensions:
            continue
        for key in {path.name.lower(), path.stem.lower()}:
            previous = index.get(key)
            if previous is None or path.stat().st_size > previous.stat().st_size:
                index[key] = path

    if not index:
        raise RuntimeError(f"No supported image files were found under {image_root}")
    return index


def resolve_image_path(image_id: object, image_index: Dict[str, Path]) -> Optional[Path]:
    raw = str(image_id).strip()
    candidate = Path(raw)
    for key in (raw.lower(), candidate.name.lower(), candidate.stem.lower()):
        if key in image_index:
            return image_index[key]
    return None


def build_vindr_pairs(config: Config) -> pd.DataFrame:
    frame = pd.read_csv(config.vindr_annotations_csv, low_memory=False)
    if config.vindr_source_split is not None:
        split_column = find_column(frame, ["split"])
        split_value = str(config.vindr_source_split).lower().strip()
        frame = frame[frame[split_column].astype(str).str.lower().str.strip() == split_value].copy()

    image_column = find_column(frame, ["image_id", "image"])
    study_column = find_column(frame, ["study_id", "series_id", "study"])
    patient_column = find_column(frame, ["patient_id", "patient"], required=False)
    laterality_column = find_column(frame, ["laterality", "breast_laterality"])
    view_column = find_column(frame, ["view", "view_position"])
    birads_column = find_column(frame, ["breast_birads", "birads", "breast_birads_assessment"])

    image_index = build_image_index(config.vindr_image_root)
    data = frame.copy()
    data["image_path"] = data[image_column].map(lambda value: resolve_image_path(value, image_index))
    data = data.dropna(subset=["image_path"]).copy()
    data["study_key"] = data[study_column].astype(str)
    data["patient_key"] = data[patient_column].astype(str) if patient_column else data["study_key"]
    data["laterality_key"] = data[laterality_column].astype(str).str.upper().str.strip()
    data["view_key"] = data[view_column].map(normalize_view)
    data["birads"] = data[birads_column].map(parse_birads)
    data["label"] = (data["birads"].fillna(0).astype(int) >= config.vindr_positive_birads_threshold).astype(int)
    data = data[data["view_key"].isin(["CC", "MLO"])].copy()
    data["pair_id"] = data["study_key"] + "_" + data["laterality_key"]
    data["split_group"] = data["patient_key"] if config.split_level == "patient" else data["study_key"]

    quality_cache: Dict[str, float] = {}

    def quality(path: Path) -> float:
        key = str(path)
        if key not in quality_cache:
            quality_cache[key] = foreground_area(crop_breast(read_image(path), config.image_size))
        return quality_cache[key]

    pairs = []
    for pair_id, group in tqdm(data.groupby("pair_id", sort=False), total=data["pair_id"].nunique(), desc="pairing VinDr"):
        cc = group[group["view_key"] == "CC"]
        mlo = group[group["view_key"] == "MLO"]
        if cc.empty or mlo.empty:
            continue

        cc_row = max(cc.itertuples(index=False), key=lambda row: quality(getattr(row, "image_path")))
        mlo_row = max(mlo.itertuples(index=False), key=lambda row: quality(getattr(row, "image_path")))
        pairs.append(
            {
                "pair_id": pair_id,
                "split_group": str(group["split_group"].iloc[0]),
                "study_id": str(group["study_key"].iloc[0]),
                "laterality": str(group["laterality_key"].iloc[0]),
                "cc_path": str(getattr(cc_row, "image_path")),
                "mlo_path": str(getattr(mlo_row, "image_path")),
                "label": int(group["label"].max()),
            }
        )

    pairs = pd.DataFrame(pairs)
    if pairs.empty:
        raise RuntimeError("No complete VinDr CC/MLO pairs were created")
    return pairs.reset_index(drop=True)


CBIS_UID_PATTERN = re.compile(r"(1\.3\.6\.1\.4\.1\.9590\.100\.1\.2\.\d+)")


def parse_cbis_uids(value: object) -> Tuple[Optional[str], Optional[str]]:
    matches = CBIS_UID_PATTERN.findall(str(value).replace("\\", "/"))
    if len(matches) < 2:
        return None, None
    return matches[0], matches[1]


def resolve_cbis_relative_path(root: Path, value: object) -> Optional[Path]:
    raw = str(value).replace("\\", "/").strip().lstrip("./").lstrip("/")
    parts = list(Path(raw).parts)
    if parts and parts[0].lower() == "cbis-ddsm":
        parts = parts[1:]
    candidate = root.joinpath(*parts)
    if candidate.exists():
        return candidate
    return None


def select_cbis_image_column(frame: pd.DataFrame, image_source: str) -> str:
    source = str(image_source).lower().strip()
    if source == "dicom":
        candidates = ["file_path", "dicom_path"]
    elif source in {"jpeg", "jpg", "png"}:
        candidates = ["image_path", "jpeg_path", "png_path"]
    else:
        raise ValueError("cbis_image_source must be 'dicom' or 'jpeg'")

    column = find_column(frame, candidates, required=False)
    if column is not None:
        return column

    for candidate in frame.columns:
        name = normalize_column_name(candidate)
        if source == "dicom" and ("file" in name or "dicom" in name) and "path" in name:
            return candidate
        if source != "dicom" and ("image" in name or "jpeg" in name or "png" in name) and "path" in name:
            return candidate
    raise KeyError(f"No {source} path column was found in dicom_info.csv")


def build_cbis_uid_index(config: Config, dicom_info_csv: Path) -> Dict[Tuple[str, str], Path]:
    frame = pd.read_csv(dicom_info_csv, low_memory=False)
    study_column = find_column(frame, ["StudyInstanceUID"])
    series_column = find_column(frame, ["SeriesInstanceUID"])
    path_column = select_cbis_image_column(frame, config.cbis_image_source)
    instance_column = find_column(frame, ["InstanceNumber"], required=False)
    mapping: Dict[Tuple[str, str], Path] = {}

    grouped = frame.groupby([study_column, series_column], sort=False)
    for (study_uid, series_uid), group in tqdm(grouped, total=grouped.ngroups, desc="indexing CBIS"):
        if instance_column is not None:
            group = group.sort_values(instance_column, kind="stable")
        for value in group[path_column]:
            path = resolve_cbis_relative_path(config.cbis_root, value)
            if path is not None:
                mapping[(str(study_uid), str(series_uid))] = path
                break

    if not mapping:
        raise RuntimeError("No CBIS-DDSM image paths were resolved from dicom_info.csv")
    return mapping


def build_cbis_pairs(config: Config) -> pd.DataFrame:
    train_csv, test_csv, dicom_info_csv = resolve_cbis_paths(config)
    frames = [pd.read_csv(train_csv, low_memory=False), pd.read_csv(test_csv, low_memory=False)]
    frame = pd.concat(frames, ignore_index=True)

    patient_column = find_column(frame, ["patient_id"])
    laterality_column = find_column(frame, ["left or right breast", "laterality"])
    view_column = find_column(frame, ["image view", "view"])
    abnormality_column = find_column(frame, ["abnormality id", "abnormality_id"])
    pathology_column = find_column(frame, ["pathology"])
    image_path_column = find_image_path_column(frame)
    uid_index = build_cbis_uid_index(config, dicom_info_csv)

    data = frame.copy()
    data["uid_pair"] = data[image_path_column].map(parse_cbis_uids)
    data["image_path"] = data["uid_pair"].map(lambda value: uid_index.get(value) if value[0] is not None else None)
    data = data.dropna(subset=["image_path"]).copy()
    data["patient_key"] = data[patient_column].astype(str)
    data["laterality_key"] = data[laterality_column].astype(str).str.upper().str.strip()
    data["view_key"] = data[view_column].map(normalize_view)
    data["abnormality_key"] = data[abnormality_column].astype(str).str.strip()
    data["label"] = data[pathology_column].map(pathology_label)
    data = data[data["view_key"].isin(["CC", "MLO"])].copy()
    data["pair_id"] = data["patient_key"] + "_" + data["laterality_key"] + "_" + data["abnormality_key"]
    data["split_group"] = data["patient_key"]

    quality_cache: Dict[str, float] = {}

    def quality(path: Path) -> float:
        key = str(path)
        if key not in quality_cache:
            quality_cache[key] = foreground_area(crop_breast(read_image(path), config.image_size))
        return quality_cache[key]

    pairs = []
    for pair_id, group in tqdm(data.groupby("pair_id", sort=False), total=data["pair_id"].nunique(), desc="pairing CBIS-DDSM"):
        cc = group[group["view_key"] == "CC"]
        mlo = group[group["view_key"] == "MLO"]
        if cc.empty or mlo.empty:
            continue

        cc_row = max(cc.itertuples(index=False), key=lambda row: quality(getattr(row, "image_path")))
        mlo_row = max(mlo.itertuples(index=False), key=lambda row: quality(getattr(row, "image_path")))
        pairs.append(
            {
                "pair_id": pair_id,
                "split_group": str(group["split_group"].iloc[0]),
                "patient_id": str(group["patient_key"].iloc[0]),
                "laterality": str(group["laterality_key"].iloc[0]),
                "abnormality_id": str(group["abnormality_key"].iloc[0]),
                "cc_path": str(getattr(cc_row, "image_path")),
                "mlo_path": str(getattr(mlo_row, "image_path")),
                "label": int(group["label"].max()),
            }
        )

    pairs = pd.DataFrame(pairs)
    if pairs.empty:
        raise RuntimeError("No complete CBIS-DDSM mass CC/MLO pairs were created")
    return pairs.reset_index(drop=True)


def build_pairs(config: Config) -> pd.DataFrame:
    if config.dataset_name == "vindr":
        return build_vindr_pairs(config)
    return build_cbis_pairs(config)


def grouped_holdout(
    pairs: pd.DataFrame,
    fraction: float,
    seed: int,
    group_column: str = "split_group",
    label_column: str = "label",
    trials: int = 256,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not 0.0 < fraction < 1.0:
        raise ValueError("fraction must be between zero and one")
    if pairs[group_column].nunique() < 2:
        raise ValueError("At least two split groups are required")

    groups = pairs[group_column].to_numpy()
    labels = pairs[label_column].to_numpy()
    total_rate = float(labels.mean())
    best = None
    best_score = float("inf")
    splitter = GroupShuffleSplit(n_splits=trials, test_size=fraction, random_state=seed)

    for keep_index, holdout_index in splitter.split(pairs, labels, groups):
        holdout_labels = labels[holdout_index]
        score = abs(len(holdout_index) / len(pairs) - fraction) + abs(float(holdout_labels.mean()) - total_rate)
        if len(np.unique(labels)) == 2 and len(np.unique(holdout_labels)) < 2:
            score += 1.0
        if score < best_score:
            best_score = score
            best = (keep_index, holdout_index)

    if best is None:
        raise RuntimeError("Unable to create a grouped split")

    keep_index, holdout_index = best
    return pairs.iloc[keep_index].reset_index(drop=True), pairs.iloc[holdout_index].reset_index(drop=True)


def make_splits(pairs: pd.DataFrame, config: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_dev, test = grouped_holdout(pairs, config.test_fraction, config.seed)
    train, validation = grouped_holdout(train_dev, config.validation_fraction, config.seed + 1)

    split_groups = [set(part["split_group"]) for part in (train, validation, test)]
    if split_groups[0] & split_groups[1] or split_groups[0] & split_groups[2] or split_groups[1] & split_groups[2]:
        raise RuntimeError("Group leakage was detected in the split")

    return train, validation, test

In [ ]:
class PairedMammographyDataset(Dataset):
    def __init__(self, pairs: pd.DataFrame, image_size: int, augment: bool) -> None:
        self.pairs = pairs.reset_index(drop=True).copy()
        self.image_size = int(image_size)
        self.augment = bool(augment)

    def __len__(self) -> int:
        return len(self.pairs)

    def augment_pair(self, cc: np.ndarray, mlo: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if random.random() < 0.5:
            cc = np.fliplr(cc).copy()
            mlo = np.fliplr(mlo).copy()

        if random.random() < 0.5:
            angle = random.uniform(-7.0, 7.0)
            translate = (
                int(round(random.uniform(-0.02, 0.02) * self.image_size)),
                int(round(random.uniform(-0.02, 0.02) * self.image_size)),
            )
            scale = random.uniform(0.97, 1.03)
            shear = [random.uniform(-3.0, 3.0), 0.0]
            cc_tensor = torch.from_numpy(cc.copy()).unsqueeze(0)
            mlo_tensor = torch.from_numpy(mlo.copy()).unsqueeze(0)
            cc = TF.affine(
                cc_tensor,
                angle=angle,
                translate=translate,
                scale=scale,
                shear=shear,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            ).squeeze(0).numpy()
            mlo = TF.affine(
                mlo_tensor,
                angle=angle,
                translate=translate,
                scale=scale,
                shear=shear,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            ).squeeze(0).numpy()

        return cc.astype(np.float32), mlo.astype(np.float32)

    @staticmethod
    def image_to_tensor(image: np.ndarray) -> torch.Tensor:
        image = np.ascontiguousarray(image.astype(np.float32))
        tensor = torch.from_numpy(image).unsqueeze(0).repeat(3, 1, 1)
        return (tensor - 0.5) / 0.5

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        row = self.pairs.iloc[index]
        cc = crop_breast(read_image(Path(row["cc_path"])), self.image_size)
        mlo = crop_breast(read_image(Path(row["mlo_path"])), self.image_size)

        if self.augment:
            cc, mlo = self.augment_pair(cc, mlo)

        return (
            self.image_to_tensor(cc),
            self.image_to_tensor(mlo),
            torch.ones(2, dtype=torch.float32),
            torch.tensor(int(row["label"]), dtype=torch.long),
        )


def worker_seed(worker_id: int) -> None:
    worker_seed_value = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed_value)
    random.seed(worker_seed_value)


def make_loaders(
    train_pairs: pd.DataFrame,
    validation_pairs: pd.DataFrame,
    test_pairs: pd.DataFrame,
    config: Config,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_dataset = PairedMammographyDataset(train_pairs, config.image_size, config.use_augmentation)
    validation_dataset = PairedMammographyDataset(validation_pairs, config.image_size, False)
    test_dataset = PairedMammographyDataset(test_pairs, config.image_size, False)

    generator = torch.Generator()
    generator.manual_seed(config.seed)
    options = {
        "batch_size": config.batch_size,
        "num_workers": config.num_workers,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": worker_seed if config.num_workers > 0 else None,
        "generator": generator,
    }
    if config.num_workers > 0:
        options["persistent_workers"] = True

    train_loader = DataLoader(train_dataset, shuffle=True, drop_last=False, **options)
    validation_loader = DataLoader(validation_dataset, shuffle=False, drop_last=False, **options)
    test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **options)
    return train_loader, validation_loader, test_loader

In [ ]:
def group_count(channels: int) -> int:
    for groups in range(min(32, channels), 0, -1):
        if channels % groups == 0:
            return groups
    return 1


def dirichlet_mean(alpha: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    alpha = alpha.float().clamp_min(eps)
    return alpha / alpha.sum(dim=-1, keepdim=True).clamp_min(eps)


def dirichlet_uncertainty(alpha: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    alpha = alpha.float().clamp_min(eps)
    strength = alpha.sum(dim=-1).clamp_min(eps)
    return alpha.new_tensor(float(alpha.shape[-1])) / strength


def classwise_js_divergence(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    p = p.float().clamp_min(eps)
    q = q.float().clamp_min(eps)
    p = p / p.sum(dim=-1, keepdim=True).clamp_min(eps)
    q = q / q.sum(dim=-1, keepdim=True).clamp_min(eps)
    midpoint = 0.5 * (p + q)
    return 0.5 * (p * (p.log() - midpoint.log()) + q * (q.log() - midpoint.log()))


def foreground_mask(x: torch.Tensor, threshold: float) -> torch.Tensor:
    image = (x * 0.5 + 0.5).clamp(0.0, 1.0)
    return (image.mean(dim=1, keepdim=True) > threshold).float()


class FeatureRefiner(nn.Module):
    def __init__(self, channels: int, se_ratio: int = 8, dilations: Tuple[int, ...] = (1, 2, 4)) -> None:
        super().__init__()
        hidden = max(8, channels // se_ratio)
        self.norm = nn.GroupNorm(group_count(channels), channels)
        self.channel_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid(),
        )
        self.context = nn.ModuleList(
            [
                nn.Conv2d(
                    channels,
                    channels,
                    kernel_size=3,
                    padding=dilation,
                    dilation=dilation,
                    groups=channels,
                    bias=False,
                )
                for dilation in dilations
            ]
        )
        self.context_fuse = nn.Sequential(
            nn.Conv2d(channels * len(dilations), channels, kernel_size=1, bias=False),
            nn.GELU(),
        )
        self.update = nn.Sequential(
            nn.Conv2d(channels, 2 * channels, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Conv2d(2 * channels, channels, kernel_size=1, bias=False),
        )
        self.gate = nn.Sequential(nn.Conv2d(channels, channels, kernel_size=1), nn.Sigmoid())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normalized = self.norm(x)
        recalibrated = normalized * self.channel_gate(normalized)
        context = torch.cat([branch(recalibrated) for branch in self.context], dim=1)
        context = self.context_fuse(context)
        return x + self.gate(context) * self.update(context)


class TokenFeedback(nn.Module):
    def __init__(self, channels: int, smooth_kernel: int = 3) -> None:
        super().__init__()
        padding = smooth_kernel // 2
        self.token_norm = nn.LayerNorm(channels)
        self.token_projection = nn.Sequential(
            nn.Linear(channels, channels, bias=False),
            nn.GELU(),
            nn.Linear(channels, channels, bias=False),
        )
        self.smooth = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=smooth_kernel, padding=padding, groups=channels, bias=False),
            nn.GELU(),
            nn.Conv2d(channels, channels, kernel_size=1, bias=False),
        )
        self.update = nn.Sequential(nn.Conv2d(channels, channels, kernel_size=1, bias=False), nn.GELU())
        self.gate = nn.Sequential(nn.Conv2d(channels, channels, kernel_size=1), nn.Sigmoid())

    def forward(
        self,
        features: torch.Tensor,
        tokens: torch.Tensor,
        indices: torch.Tensor,
        attention: torch.Tensor,
    ) -> torch.Tensor:
        batch_size, channels, height, width = features.shape
        projected = self.token_projection(self.token_norm(tokens))
        weights = attention / attention.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        projected = projected * weights.unsqueeze(-1)
        sparse = features.new_zeros(batch_size, channels, height * width)
        expanded_indices = indices.unsqueeze(1).expand(-1, channels, -1)
        sparse.scatter_add_(2, expanded_indices, projected.transpose(1, 2))
        feedback = self.smooth(sparse.view(batch_size, channels, height, width))
        update = self.update(feedback)
        return features + self.gate(feedback) * update


class DiverseTokenSelector(nn.Module):
    def __init__(
        self,
        channels: int,
        num_tokens: int,
        candidate_factor: int,
        minimum_foreground_coverage: float,
    ) -> None:
        super().__init__()
        self.num_tokens = int(num_tokens)
        self.candidate_factor = int(candidate_factor)
        self.minimum_foreground_coverage = float(minimum_foreground_coverage)
        self.scorer = nn.Conv2d(channels, 1, kernel_size=1)

    @staticmethod
    def gather_tokens(features: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
        batch_size, channels, _, _ = features.shape
        flattened = features.flatten(2)
        expanded_indices = indices.unsqueeze(1).expand(batch_size, channels, -1)
        return torch.gather(flattened, 2, expanded_indices).transpose(1, 2).contiguous()

    @staticmethod
    def farthest_point_indices(candidates: torch.Tensor, height: int, width: int, count: int) -> torch.Tensor:
        if candidates.numel() == 0:
            raise RuntimeError("Token selection received no candidates")

        coordinates = torch.stack([candidates // width, candidates % width], dim=1).float()
        selected = [0]
        minimum_distance = torch.full((candidates.numel(),), float("inf"), device=candidates.device)

        while len(selected) < min(count, candidates.numel()):
            last = coordinates[selected[-1]].unsqueeze(0)
            distance = ((coordinates - last) ** 2).sum(dim=1)
            minimum_distance = torch.minimum(minimum_distance, distance)
            selected.append(int(minimum_distance.argmax().item()))

        selected_indices = candidates[torch.tensor(selected, device=candidates.device)]
        if selected_indices.numel() < count:
            selected_indices = torch.cat(
                [selected_indices, selected_indices[-1:].repeat(count - selected_indices.numel())]
            )
        return selected_indices

    def forward(self, features: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch_size, _, height, width = features.shape
        spatial_size = height * width
        logits = self.scorer(features).flatten(1)
        mask_flat = mask.flatten(1) > 0.5
        coverage = mask_flat.float().mean(dim=1)
        all_positions = torch.arange(spatial_size, device=features.device)
        selected = []

        for batch_index in range(batch_size):
            valid = mask_flat[batch_index] if coverage[batch_index] >= self.minimum_foreground_coverage else torch.ones_like(mask_flat[batch_index])
            valid_indices = all_positions[valid]
            candidate_count = min(valid_indices.numel(), max(self.num_tokens, self.candidate_factor * self.num_tokens))
            ranked = logits[batch_index, valid_indices].topk(candidate_count, largest=True).indices
            candidates = valid_indices[ranked]
            selected.append(self.farthest_point_indices(candidates, height, width, self.num_tokens))

        indices = torch.stack(selected, dim=0)
        masked_logits = logits.masked_fill(~mask_flat & (coverage >= self.minimum_foreground_coverage).unsqueeze(1), -1e9)
        attention = torch.softmax(masked_logits, dim=1).gather(1, indices)
        tokens = self.gather_tokens(features, indices)
        return tokens, attention, indices


class TokenEvidenceHead(nn.Module):
    def __init__(self, channels: int, num_classes: int, evidence_floor: float) -> None:
        super().__init__()
        hidden = max(64, channels // 2)
        self.network = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, num_classes),
        )
        self.evidence_floor = float(evidence_floor)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        evidence = F.softplus(self.network(tokens.float())) + self.evidence_floor
        return evidence.sum(dim=1) + 1.0


class ConvNeXtEncoder(nn.Module):
    def __init__(self, variant: str, pretrained: bool) -> None:
        super().__init__()
        variant = variant.lower().strip()
        if variant == "tiny":
            weights = torchvision.models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None
            model = torchvision.models.convnext_tiny(weights=weights)
            self.out_channels = 768
        elif variant == "small":
            weights = torchvision.models.ConvNeXt_Small_Weights.IMAGENET1K_V1 if pretrained else None
            model = torchvision.models.convnext_small(weights=weights)
            self.out_channels = 768
        else:
            raise ValueError("backbone_variant must be 'tiny' or 'small'")
        self.features = model.features

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


class ConflictAwareFusion(nn.Module):
    def __init__(self, num_classes: int, gamma_init: float) -> None:
        super().__init__()
        self.num_classes = int(num_classes)
        self.log_gamma = nn.Parameter(torch.tensor(float(gamma_init)).log())

    def gamma(self) -> torch.Tensor:
        return self.log_gamma.exp().clamp(1e-4, 50.0)

    def forward(
        self,
        alpha_cc: torch.Tensor,
        alpha_mlo: torch.Tensor,
        view_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        if view_mask is None:
            view_mask = torch.ones(alpha_cc.size(0), 2, device=alpha_cc.device, dtype=alpha_cc.dtype)
        view_mask = view_mask.float().clamp(0.0, 1.0)

        alpha_cc = alpha_cc.float().clamp_min(1e-6)
        alpha_mlo = alpha_mlo.float().clamp_min(1e-6)
        p_cc = dirichlet_mean(alpha_cc)
        p_mlo = dirichlet_mean(alpha_mlo)
        strength_cc = alpha_cc.sum(dim=-1)
        strength_mlo = alpha_mlo.sum(dim=-1)
        classes = strength_cc.new_tensor(float(self.num_classes))
        reliability_cc = strength_cc / (strength_cc + classes)
        reliability_mlo = strength_mlo / (strength_mlo + classes)
        reliability_cc = reliability_cc * view_mask[:, 0]
        reliability_mlo = reliability_mlo * view_mask[:, 1]
        denominator = (reliability_cc + reliability_mlo).clamp_min(1e-8)
        weight_cc = reliability_cc / denominator
        weight_mlo = reliability_mlo / denominator

        js_k = classwise_js_divergence(p_cc, p_mlo)
        both_present = (view_mask[:, 0] > 0.5) & (view_mask[:, 1] > 0.5)
        js_k = torch.where(both_present.unsqueeze(1), js_k, torch.zeros_like(js_k))
        gate_k = torch.exp(-self.gamma() * js_k)
        evidence_cc = (alpha_cc - 1.0).clamp_min(0.0)
        evidence_mlo = (alpha_mlo - 1.0).clamp_min(0.0)
        pooled_evidence = weight_cc.unsqueeze(1) * evidence_cc + weight_mlo.unsqueeze(1) * evidence_mlo
        fused_evidence = gate_k * pooled_evidence
        no_views = view_mask.sum(dim=1) == 0
        fused_evidence = torch.where(no_views.unsqueeze(1), torch.zeros_like(fused_evidence), fused_evidence)
        alpha = fused_evidence + 1.0

        return alpha, {
            "p_cc": p_cc,
            "p_mlo": p_mlo,
            "u_cc": dirichlet_uncertainty(alpha_cc),
            "u_mlo": dirichlet_uncertainty(alpha_mlo),
            "weight_cc": weight_cc,
            "weight_mlo": weight_mlo,
            "js_k": js_k,
            "gate_k": gate_k,
            "js": js_k.sum(dim=1),
            "gate": gate_k.mean(dim=1),
        }


class CEINet(nn.Module):
    def __init__(self, config: Config) -> None:
        super().__init__()
        self.config = config
        self.encoder = ConvNeXtEncoder(config.backbone_variant, config.pretrained_backbone)
        channels = self.encoder.out_channels
        self.refiner = FeatureRefiner(channels)
        self.selector = DiverseTokenSelector(
            channels,
            config.num_tokens,
            config.token_candidate_factor,
            config.minimum_foreground_coverage,
        )
        self.feedback = TokenFeedback(channels)
        self.evidence_head = TokenEvidenceHead(channels, config.num_classes, config.evidence_floor)
        self.fusion = ConflictAwareFusion(config.num_classes, config.conflict_gamma_init)

    def encode_view(self, x: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        features = self.refiner(self.encoder(x))
        mask = F.interpolate(
            foreground_mask(x, self.config.foreground_threshold),
            size=features.shape[-2:],
            mode="nearest",
        )
        tokens, attention, indices = self.selector(features, mask)

        for _ in range(self.config.feedback_steps):
            features = self.feedback(features, tokens, indices, attention)
            tokens = self.selector.gather_tokens(features, indices)

        alpha = self.evidence_head(tokens)

        return alpha, {
            "token_attention": attention,
            "token_indices": indices,
            "foreground_mask": mask,
        }

    def forward(
        self,
        cc: torch.Tensor,
        mlo: torch.Tensor,
        view_mask: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        alpha_cc, cc_auxiliary = self.encode_view(cc)
        alpha_mlo, mlo_auxiliary = self.encode_view(mlo)
        alpha, fusion_auxiliary = self.fusion(alpha_cc, alpha_mlo, view_mask)
        return {
            "alpha": alpha,
            "alpha_cc": alpha_cc,
            "alpha_mlo": alpha_mlo,
            "p": dirichlet_mean(alpha),
            "u": dirichlet_uncertainty(alpha),
            **fusion_auxiliary,
            "cc_token_attention": cc_auxiliary["token_attention"],
            "mlo_token_attention": mlo_auxiliary["token_attention"],
            "cc_token_indices": cc_auxiliary["token_indices"],
            "mlo_token_indices": mlo_auxiliary["token_indices"],
            "cc_foreground_mask": cc_auxiliary["foreground_mask"],
            "mlo_foreground_mask": mlo_auxiliary["foreground_mask"],
        }

In [ ]:
def kl_dirichlet_to_uniform(alpha: torch.Tensor) -> torch.Tensor:
    alpha = alpha.float().clamp_min(1e-8)
    prior = torch.ones_like(alpha)
    alpha_sum = alpha.sum(dim=-1, keepdim=True)
    prior_sum = prior.sum(dim=-1, keepdim=True)
    log_beta_alpha = torch.lgamma(alpha_sum) - torch.lgamma(alpha).sum(dim=-1, keepdim=True)
    log_beta_prior = torch.lgamma(prior_sum) - torch.lgamma(prior).sum(dim=-1, keepdim=True)
    divergence = ((alpha - prior) * (torch.digamma(alpha) - torch.digamma(alpha_sum))).sum(dim=-1, keepdim=True)
    return (divergence + log_beta_alpha - log_beta_prior).squeeze(1)


def evidential_cross_entropy(
    alpha: torch.Tensor,
    labels: torch.Tensor,
    epoch: int,
    config: Config,
    sample_weights: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    alpha = alpha.float().clamp_min(1e-6)
    targets = F.one_hot(labels, num_classes=config.num_classes).float()
    strength = alpha.sum(dim=-1, keepdim=True)
    classification = (targets * (torch.digamma(strength) - torch.digamma(alpha))).sum(dim=-1)
    annealing = min(1.0, epoch / max(1, config.kl_anneal_epochs))
    loss = classification + config.kl_weight * annealing * kl_dirichlet_to_uniform(alpha)

    if sample_weights is None:
        return loss.mean()
    return (loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-6)


def gate_calibration_loss(js_k: torch.Tensor, gate_k: torch.Tensor, view_mask: torch.Tensor) -> torch.Tensor:
    target = torch.exp(-js_k / math.log(2.0))
    error = (gate_k - target).pow(2).mean(dim=1)
    both_present = ((view_mask[:, 0] > 0.5) & (view_mask[:, 1] > 0.5)).float()
    return (error * both_present).sum() / both_present.sum().clamp_min(1.0)


def compute_loss(
    output: Dict[str, torch.Tensor],
    labels: torch.Tensor,
    epoch: int,
    config: Config,
    positive_weight: float,
    view_mask: torch.Tensor,
) -> Dict[str, torch.Tensor]:
    sample_weights = torch.where(
        labels == 1,
        torch.full_like(labels, float(positive_weight), dtype=torch.float32),
        torch.ones_like(labels, dtype=torch.float32),
    )
    fused = evidential_cross_entropy(output["alpha"], labels, epoch, config, sample_weights)
    cc = evidential_cross_entropy(output["alpha_cc"], labels, epoch, config, sample_weights)
    mlo = evidential_cross_entropy(output["alpha_mlo"], labels, epoch, config, sample_weights)
    auxiliary = 0.5 * config.auxiliary_view_weight * (cc + mlo)
    calibration = gate_calibration_loss(output["js_k"], output["gate_k"], view_mask)
    total = fused + auxiliary + config.gate_calibration_weight * calibration
    return {
        "total": total,
        "fused": fused.detach(),
        "auxiliary": auxiliary.detach(),
        "calibration": calibration.detach(),
    }


def expected_calibration_error(labels: np.ndarray, probabilities: np.ndarray, bins: int) -> float:
    labels = np.asarray(labels, dtype=np.float64)
    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64), 0.0, 1.0)
    boundaries = np.linspace(0.0, 1.0, bins + 1)
    calibration_error = 0.0

    for index in range(bins):
        lower = boundaries[index]
        upper = boundaries[index + 1]
        in_bin = (probabilities >= lower) & (probabilities < upper)
        if index == bins - 1:
            in_bin = (probabilities >= lower) & (probabilities <= upper)
        if not in_bin.any():
            continue
        confidence = float(probabilities[in_bin].mean())
        accuracy = float(labels[in_bin].mean())
        calibration_error += float(in_bin.mean()) * abs(accuracy - confidence)

    return float(calibration_error)


def binary_nll(labels: np.ndarray, probabilities: np.ndarray) -> float:
    labels = np.asarray(labels, dtype=np.float64)
    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64), 1e-8, 1.0 - 1e-8)
    return float(-(labels * np.log(probabilities) + (1.0 - labels) * np.log(1.0 - probabilities)).mean())


def binary_metrics(labels: np.ndarray, probabilities: np.ndarray, bins: int) -> Dict[str, float]:
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64), 0.0, 1.0)
    auroc = float(roc_auc_score(labels, probabilities)) if np.unique(labels).size == 2 else float("nan")
    return {
        "auroc": auroc,
        "ece": expected_calibration_error(labels, probabilities, bins),
        "nll": binary_nll(labels, probabilities),
        "brier": float(np.mean((probabilities - labels) ** 2)),
    }


class TemperatureScaler(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.log_temperature.exp().clamp(1e-3, 1e3)


def probability_to_logit(probability: torch.Tensor) -> torch.Tensor:
    probability = probability.float().clamp(1e-6, 1.0 - 1e-6)
    return torch.log(probability) - torch.log1p(-probability)


def apply_temperature(probability: torch.Tensor, temperature: float) -> torch.Tensor:
    return torch.sigmoid(probability_to_logit(probability) / max(float(temperature), 1e-6))


def fit_temperature(probabilities: np.ndarray, labels: np.ndarray, device: torch.device) -> float:
    logits = probability_to_logit(torch.as_tensor(probabilities, device=device)).detach()
    targets = torch.as_tensor(labels, dtype=torch.float32, device=device)
    scaler = TemperatureScaler().to(device)
    optimizer = torch.optim.LBFGS([scaler.log_temperature], lr=0.1, max_iter=50, line_search_fn="strong_wolfe")
    criterion = nn.BCEWithLogitsLoss()

    def closure() -> torch.Tensor:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(scaler(logits).reshape(-1), targets.reshape(-1))
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(scaler.log_temperature.exp().clamp(1e-3, 1e3).item())

In [ ]:
def prepare_batch(
    batch: Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
    device: torch.device,
    retained_view: Optional[str] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    cc, mlo, view_mask, labels = [item.to(device, non_blocking=True) for item in batch]

    if retained_view == "cc":
        mlo = torch.zeros_like(mlo)
        view_mask[:, 1] = 0.0
    elif retained_view == "mlo":
        cc = torch.zeros_like(cc)
        view_mask[:, 0] = 0.0

    return cc, mlo, view_mask, labels


def train_epoch(
    model: CEINet,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    config: Config,
    device: torch.device,
    positive_weight: float,
    scaler: torch.cuda.amp.GradScaler,
) -> Dict[str, float]:
    model.train()
    amp_enabled = config.amp and device.type == "cuda"
    totals = {"total": 0.0, "fused": 0.0, "auxiliary": 0.0, "calibration": 0.0}
    seen = 0

    for batch in tqdm(loader, desc=f"train {epoch:03d}", leave=False):
        cc, mlo, view_mask, labels = prepare_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)

        if amp_enabled:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                output = model(cc, mlo, view_mask)
            loss_values = compute_loss(output, labels, epoch, config, positive_weight, view_mask)
            loss = loss_values["total"]
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            output = model(cc, mlo, view_mask)
            loss_values = compute_loss(output, labels, epoch, config, positive_weight, view_mask)
            loss = loss_values["total"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
            optimizer.step()

        batch_size = labels.size(0)
        seen += batch_size
        for name in totals:
            totals[name] += float(loss_values[name].item()) * batch_size

    return {name: value / max(seen, 1) for name, value in totals.items()}


@torch.inference_mode()
def collect_predictions(
    model: CEINet,
    loader: DataLoader,
    config: Config,
    device: torch.device,
    temperature: Optional[float] = None,
    retained_view: Optional[str] = None,
) -> Dict[str, np.ndarray]:
    model.eval()
    output_lists: Dict[str, list] = {
        "label": [],
        "probability": [],
        "probability_cc": [],
        "probability_mlo": [],
        "uncertainty": [],
        "uncertainty_cc": [],
        "uncertainty_mlo": [],
        "js": [],
        "gate": [],
        "weight_cc": [],
        "weight_mlo": [],
        "js_k": [],
        "gate_k": [],
    }

    for batch in tqdm(loader, desc="evaluate", leave=False):
        cc, mlo, view_mask, labels = prepare_batch(batch, device, retained_view)
        output = model(cc, mlo, view_mask)
        probability = output["p"][:, 1]
        if temperature is not None:
            probability = apply_temperature(probability, temperature)

        values = {
            "label": labels,
            "probability": probability,
            "probability_cc": output["p_cc"][:, 1],
            "probability_mlo": output["p_mlo"][:, 1],
            "uncertainty": output["u"],
            "uncertainty_cc": output["u_cc"],
            "uncertainty_mlo": output["u_mlo"],
            "js": output["js"],
            "gate": output["gate"],
            "weight_cc": output["weight_cc"],
            "weight_mlo": output["weight_mlo"],
            "js_k": output["js_k"],
            "gate_k": output["gate_k"],
        }
        for key, value in values.items():
            output_lists[key].append(value.detach().cpu().numpy())

    return {key: np.concatenate(values, axis=0) for key, values in output_lists.items()}


def summarize_predictions(predictions: Dict[str, np.ndarray], config: Config) -> Dict[str, float]:
    summary = binary_metrics(predictions["label"], predictions["probability"], config.ece_bins)
    summary.update(
        {
            "mean_uncertainty": float(predictions["uncertainty"].mean()),
            "mean_js": float(predictions["js"].mean()),
            "mean_gate": float(predictions["gate"].mean()),
            "mean_weight_cc": float(predictions["weight_cc"].mean()),
            "mean_weight_mlo": float(predictions["weight_mlo"].mean()),
        }
    )
    return summary


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def temperature_scaled_predictions(
    predictions: Dict[str, np.ndarray],
    temperature: float,
    device: torch.device,
) -> Dict[str, np.ndarray]:
    scaled = dict(predictions)
    probability = torch.as_tensor(predictions["probability"], dtype=torch.float32, device=device)
    scaled["probability"] = apply_temperature(probability, temperature).detach().cpu().numpy()
    return scaled


def fit_model(
    model: CEINet,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    test_loader: DataLoader,
    config: Config,
    device: torch.device,
    positive_weight: float,
) -> Dict[str, object]:
    config.output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = config.output_dir / "best_ceinet_s.pt"
    history_path = config.output_dir / "history.json"
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=config.amp and device.type == "cuda")
    best_score = -float("inf")
    best_epoch = 0
    stale_epochs = 0
    history = []

    for epoch in range(1, config.epochs + 1):
        training = train_epoch(model, train_loader, optimizer, epoch, config, device, positive_weight, scaler)
        scheduler.step()
        validation_predictions = collect_predictions(model, validation_loader, config, device)
        validation = summarize_predictions(validation_predictions, config)
        history.append({"epoch": epoch, "train": training, "validation": validation})

        with history_path.open("w", encoding="utf-8") as handle:
            json.dump(history, handle, indent=2)

        print(
            f"epoch={epoch:03d} "
            f"train_loss={training['total']:.5f} "
            f"val_auroc={validation['auroc']:.4f} "
            f"val_ece={validation['ece']:.4f} "
            f"val_nll={validation['nll']:.4f}"
        )

        if np.isfinite(validation["auroc"]) and validation["auroc"] > best_score + config.early_stopping_delta:
            best_score = validation["auroc"]
            best_epoch = epoch
            stale_epochs = 0
            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "epoch": epoch,
                    "validation_auroc": best_score,
                    "config": config_to_dict(config),
                },
                checkpoint_path,
            )
        else:
            stale_epochs += 1
            if stale_epochs >= config.early_stopping_patience:
                break

    if not checkpoint_path.exists():
        raise RuntimeError("Training ended before a valid checkpoint was saved")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["state_dict"])
    validation_predictions = collect_predictions(model, validation_loader, config, device)
    temperature = fit_temperature(validation_predictions["probability"], validation_predictions["label"], device)
    test_raw = collect_predictions(model, test_loader, config, device)
    test_calibrated = temperature_scaled_predictions(test_raw, temperature, device)
    test_raw_metrics = summarize_predictions(test_raw, config)
    test_calibrated_metrics = summarize_predictions(test_calibrated, config)

    result: Dict[str, object] = {
        "best_epoch": best_epoch,
        "best_validation_auroc": best_score,
        "temperature": temperature,
        "test_raw": test_raw_metrics,
        "test_temperature_scaled": test_calibrated_metrics,
    }

    if config.evaluate_missing_views:
        cc_only = summarize_predictions(collect_predictions(model, test_loader, config, device, retained_view="cc"), config)
        mlo_only = summarize_predictions(collect_predictions(model, test_loader, config, device, retained_view="mlo"), config)
        result["test_cc_only"] = cc_only
        result["test_mlo_only"] = mlo_only

    with (config.output_dir / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(result, handle, indent=2)

    np.savez(
        config.output_dir / "test_predictions.npz",
        **test_raw,
        probability_temperature_scaled=test_calibrated["probability"],
        temperature=np.asarray([temperature], dtype=np.float64),
    )

    return result

In [ ]:
def run_experiment(config: Config) -> Dict[str, object]:
    config = validate_config(config)
    config.output_dir = config.output_dir / f"ceinet_s_{config.dataset_name}"
    seed_everything(config.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pairs = build_pairs(config)
    train_pairs, validation_pairs, test_pairs = make_splits(pairs, config)
    config.output_dir.mkdir(parents=True, exist_ok=True)

    with (config.output_dir / "config.json").open("w", encoding="utf-8") as handle:
        json.dump(config_to_dict(config), handle, indent=2)

    train_pairs.to_csv(config.output_dir / "train_pairs.csv", index=False)
    validation_pairs.to_csv(config.output_dir / "validation_pairs.csv", index=False)
    test_pairs.to_csv(config.output_dir / "test_pairs.csv", index=False)

    train_loader, validation_loader, test_loader = make_loaders(
        train_pairs,
        validation_pairs,
        test_pairs,
        config,
    )
    positive_count = int((train_pairs["label"] == 1).sum())
    negative_count = int((train_pairs["label"] == 0).sum())
    positive_weight = negative_count / max(positive_count, 1)

    model = CEINet(config).to(device)
    print(
        {
            "dataset": config.dataset_name,
            "device": str(device),
            "parameters": count_parameters(model),
            "pairs": len(pairs),
            "train": len(train_pairs),
            "validation": len(validation_pairs),
            "test": len(test_pairs),
            "positive_weight": positive_weight,
        }
    )

    return fit_model(
        model,
        train_loader,
        validation_loader,
        test_loader,
        config,
        device,
        positive_weight,
    )


results = run_experiment(cfg)
print(json.dumps(results, indent=2))